# Customer Churn — XGBoost Optimization
- **Owner:** Mohamed Ali
- **Notebook:** 04_ali_xgboost_optimization.ipynb

This notebook trains and lightly optimizes an XGBoost churn model using the final Phase 2 processed dataset and the Phase 3 feature schema. It compares XGBoost against the Logistic Regression baseline using recall-first metrics, exports a deployment-safe final model pipeline, and prepares the handoff to the Streamlit app phase.

| PHASE 4 ARTIFACT CONTRACT | |
| :--- | :--- |
| **INPUTS** | `data/cleaned/processed_telco.csv`<br>`models/feature_schema.pkl`<br>`models/logistic_regression_pipeline.pkl` |
| **OUTPUTS** | `models/xgboost_pipeline.pkl`<br>`models/final_model_pipeline.pkl`<br>`reports/xgboost_model_comparison.md`<br>`assets/plots/models/*.png` |
| **NEXT** | `05_ali_streamlit_app.ipynb` |

## Inputs, Rules, and Optimization Strategy

**Inputs:**
- `data/cleaned/processed_telco.csv`
- `models/feature_schema.pkl`
- `models/logistic_regression_pipeline.pkl`
- `reports/baseline_model_comparison.md`

**Required Outputs:**
- `models/xgboost_pipeline.pkl`
- `models/final_model_pipeline.pkl`
- `reports/xgboost_model_comparison.md`
- `assets/plots/models/confusion_matrix_xgb.png`
- `assets/plots/models/roc_curve_xgb_vs_lr.png`
- `assets/plots/models/feature_importance_xgb.png`

**Rules:**
- Use `processed_telco.csv` only.
- Enforce `feature_schema.pkl`.
- No preprocessing recreation.
- No SMOTE/resampling.
- No threshold tuning.
- No Optuna, no SHAP, no MLflow, no ensembling.

**Lightweight Optimization Strategy:**
- Train one XGBoost baseline.
- Run a very small GridSearchCV on training data only.
- Use recall as the refit metric.
- Compare optimized XGBoost against Logistic Regression.

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys
import joblib
import warnings

from xgboost import XGBClassifier

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.base import clone
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
    roc_curve,
    auc
)

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 100

print("✅ Libraries loaded successfully.")

## 2. Configuration

In [ ]:
# Paths
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    candidate_root = PROJECT_ROOT.parent
    if (candidate_root / "data").exists() and (candidate_root / "notebooks").exists():
        PROJECT_ROOT = candidate_root

if not (PROJECT_ROOT / "data").exists():
    raise FileNotFoundError("Could not locate project root containing the data directory.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from runtime_audit_utils import backup_if_overwriting, record_regeneration_step
PROCESSED_DATA_PATH = PROJECT_ROOT / "data/cleaned/processed_telco.csv"
FEATURE_SCHEMA_PATH = PROJECT_ROOT / "models/feature_schema.pkl"
LR_MODEL_PATH       = PROJECT_ROOT / "models/logistic_regression_pipeline.pkl"

MODELS_PATH         = PROJECT_ROOT / "models"
REPORTS_PATH        = PROJECT_ROOT / "reports"
MODEL_PLOTS_PATH    = PROJECT_ROOT / "assets/plots/models"

XGB_MODEL_PATH      = MODELS_PATH / "xgboost_pipeline.pkl"
FINAL_MODEL_PATH    = MODELS_PATH / "final_model_pipeline.pkl"
REPORT_PATH         = REPORTS_PATH / "xgboost_model_comparison.md"

CM_XGB_PATH         = MODEL_PLOTS_PATH / "confusion_matrix_xgb.png"
ROC_COMPARE_PATH    = MODEL_PLOTS_PATH / "roc_curve_xgb_vs_lr.png"
XGB_IMPORTANCE_PATH = MODEL_PLOTS_PATH / "feature_importance_xgb.png"

# Constants
RANDOM_STATE        = 42
TARGET_COLUMN       = "Churn Label"
EXPECTED_ROWS       = 7043
EXPECTED_COLS       = 28
EXPECTED_FEATURES   = 27
TEST_SIZE           = 0.20
VALIDATION_SIZE     = 0.20
VALIDATION_SIZE     = 0.20
POSITIVE_LABEL      = 1

LEAKAGE_COLUMNS = ["Churn Score", "Churn Value", "Churn Reason", "CLTV"]

PHASE3_LR_BENCHMARK = {
    "Recall": 0.5481,
    "F1": 0.5985,
    "ROC_AUC": 0.8407
}

for p in [MODELS_PATH, REPORTS_PATH, MODEL_PLOTS_PATH]:
    p.mkdir(parents=True, exist_ok=True)

print("✅ Configuration complete.")

## 3. Load Phase 4 Inputs
We load the processed dataset, the frozen Phase 3 feature schema, and the Logistic Regression baseline pipeline.

In [3]:
df = pd.read_csv(PROCESSED_DATA_PATH)
feature_schema = joblib.load(FEATURE_SCHEMA_PATH)
lr_pipeline = joblib.load(LR_MODEL_PATH)

print(f"Data shape: {df.shape}")
print(f"Feature schema length: {len(feature_schema)}")
print(f"Target distribution:\n{df[TARGET_COLUMN].value_counts().sort_index()}")
display(df.head(3))

Data shape: (7043, 28)
Feature schema length: 27
Target distribution:
Churn Label
0    5174
1    1869
Name: count, dtype: int64


,Gender,Senior Citizen,Partner,Dependents,Tenure Months,Phone Service,Multiple Lines,Online Security,Online Backup,Device Protection,...,Num_Add_On_Services,Has_Online_Services,Avg_Monthly_Spend,Is_Long_Term_Contract,Internet Service_Fiber optic,Internet Service_No,Payment Method_Credit card (automatic),Payment Method_Electronic check,Payment Method_Mailed check,Churn Label
0,1,0,0,0,2,1,0,1,1,0,...,2,1,54.08,0,0,0,0,0,1,1
1,0,0,0,1,2,1,0,0,0,0,...,0,0,75.82,0,1,0,0,1,0,1
2,0,0,0,1,8,1,1,0,0,1,...,3,0,102.56,0,1,0,0,1,0,1


### ✅ GATE 1 — Input and Schema Validation
_This gate protects the pd.get_dummies feature schema created in Phase 2._

In [ ]:
gate1_errors = []

required_paths = [PROCESSED_DATA_PATH, FEATURE_SCHEMA_PATH, LR_MODEL_PATH]
for path in required_paths:
    if path.exists() and path.stat().st_size > 0:
        print(f"✅ GATE 1 PASS: Found {path}")
    else:
        gate1_errors.append(f"❌ GATE 1 FAIL: Missing or empty file: {path}")

if df.shape == (EXPECTED_ROWS, EXPECTED_COLS):
    print(f"✅ GATE 1 PASS: Dataset shape is {df.shape}")
else:
    gate1_errors.append(f"❌ GATE 1 FAIL: Expected {(EXPECTED_ROWS, EXPECTED_COLS)}, got {df.shape}")

if len(feature_schema) == EXPECTED_FEATURES:
    print("✅ GATE 1 PASS: feature_schema.pkl contains 27 features.")
else:
    gate1_errors.append(f"❌ GATE 1 FAIL: feature_schema length is {len(feature_schema)}")

expected_feature_order = [c for c in df.columns if c != TARGET_COLUMN]
if feature_schema == expected_feature_order:
    print("✅ GATE 1 PASS: feature_schema matches processed_telco.csv feature order.")
else:
    gate1_errors.append("❌ GATE 1 FAIL: feature_schema does not match processed_telco.csv feature order.")

if df.columns[-1] == TARGET_COLUMN and set(df[TARGET_COLUMN].unique()).issubset({0, 1}):
    print("✅ GATE 1 PASS: Target is binary and remains the last column.")
else:
    gate1_errors.append("❌ GATE 1 FAIL: Target column is invalid.")

null_counts = df.isnull().sum()
unexpected_nulls = null_counts[(null_counts > 0) & (null_counts.index != "Total Charges")]
if unexpected_nulls.empty and not df.select_dtypes(include=["object"]).columns.tolist():
    print(f"✅ GATE 1 PASS: Only {int(null_counts['Total Charges'])} expected Total Charges nulls remain and zero object dtype columns.")
else:
    gate1_errors.append(f"❌ GATE 1 FAIL: Unexpected nulls or object dtype columns found: {unexpected_nulls.to_dict()}.")

found_leakage = [c for c in LEAKAGE_COLUMNS if c in df.columns]
if not found_leakage:
    print("✅ GATE 1 PASS: No leakage columns found.")
else:
    gate1_errors.append(f"❌ GATE 1 FAIL: Leakage columns found: {found_leakage}")

if gate1_errors:
    for err in gate1_errors:
        print(err)
    raise ValueError("❌ GATE 1 FAILED. Stop and rerun upstream phases.")

print("\n🟢 GATE 1 COMPLETE — Inputs are valid.")

## 4. Create X and y From Frozen Feature Schema

In [5]:
X = df[feature_schema].copy()
y = df[TARGET_COLUMN].copy()

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")

X shape: (7043, 27)
y shape: (7043,)


## 5. Recreate Phase 3 Stratified Split
The split rule is identical to Phase 3 for fair comparison.

In [ ]:
X_development, X_test, y_development, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)

validation_fraction_of_development = VALIDATION_SIZE / (1 - TEST_SIZE)
X_train, X_validation, y_train, y_validation = train_test_split(
    X_development,
    y_development,
    test_size=validation_fraction_of_development,
    random_state=RANDOM_STATE,
    stratify=y_development
)

split_summary = pd.DataFrame({
    "Dataset": ["Full", "Train", "Validation", "Untouched Test"],
    "Rows": [len(y), len(y_train), len(y_validation), len(y_test)],
    "Churn_Rate": [y.mean(), y_train.mean(), y_validation.mean(), y_test.mean()]
})
display(split_summary)


## 6. Evaluate Phase 3 Logistic Regression Baseline
We reload the saved baseline pipeline and evaluate it on the recreated split.

In [ ]:
def evaluate_model(model_name, pipeline, X_eval, y_eval):
    y_pred = pipeline.predict(X_eval)
    y_proba = pipeline.predict_proba(X_eval)[:, 1]
    return {
        "Model": model_name,
        "Recall": recall_score(y_eval, y_pred, zero_division=0),
        "F1": f1_score(y_eval, y_pred, zero_division=0),
        "ROC_AUC": roc_auc_score(y_eval, y_proba),
        "Precision": precision_score(y_eval, y_pred, zero_division=0),
        "Accuracy": accuracy_score(y_eval, y_pred),
        "y_pred": y_pred,
        "y_proba": y_proba
    }

lr_results = evaluate_model("Logistic Regression", lr_pipeline, X_validation, y_validation)
print("Phase 3 Logistic Regression validation baseline:")
for metric in ["Recall", "F1", "ROC_AUC", "Precision", "Accuracy"]:
    print(f"{metric}: {lr_results[metric]:.4f}")

### ✅ GATE 2 — Split and Baseline Validation

In [ ]:
gate2_errors = []

if len(X_train) + len(X_validation) + len(X_test) == EXPECTED_ROWS:
    print("✅ GATE 2 PASS: Train + validation + untouched test row counts preserve all rows.")
else:
    gate2_errors.append("❌ GATE 2 FAIL: Three-way split row counts do not preserve all rows.")

if all(list(frame.columns) == feature_schema for frame in [X_train, X_validation, X_test]):
    print("✅ GATE 2 PASS: Feature order preserved after split.")
else:
    gate2_errors.append("❌ GATE 2 FAIL: Feature order changed after split.")

if all(abs(rate - y.mean()) < 0.005 for rate in [y_train.mean(), y_validation.mean(), y_test.mean()]):
    print("✅ GATE 2 PASS: Stratified churn rate preserved.")
else:
    gate2_errors.append("❌ GATE 2 FAIL: Churn rate drift detected.")

if hasattr(lr_pipeline, "predict_proba"):
    print("✅ GATE 2 PASS: Logistic Regression baseline can produce probabilities.")
else:
    gate2_errors.append("❌ GATE 2 FAIL: Logistic Regression baseline cannot produce probabilities.")

for metric, expected in PHASE3_LR_BENCHMARK.items():
    actual = lr_results[metric]
    if abs(actual - expected) <= 0.05:
        print(f"✅ GATE 2 PASS: LR validation {metric} remains consistent ({actual:.4f}).")
    else:
        print(f"⚠️ GATE 2 WARN: LR validation {metric} differs from legacy Phase 3 report ({actual:.4f} vs {expected:.4f}).")

if gate2_errors:
    for err in gate2_errors:
        print(err)
    raise ValueError("❌ GATE 2 FAILED. Fix split or baseline loading before modeling.")

print("\n🟢 GATE 2 COMPLETE — Three-way split and validation baseline are valid.")


## 7. Train XGBoost Baseline Pipeline
XGBoost receives the already numeric Phase 2 feature matrix. No scaling or preprocessing is added.

In [ ]:
xgb_baseline_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("model", XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        n_estimators=120,
        max_depth=3,
        learning_rate=0.08,
        subsample=0.9,
        colsample_bytree=0.9,
        random_state=RANDOM_STATE,
        n_jobs=1
    ))
])

xgb_baseline_pipeline.fit(X_train, y_train)
xgb_baseline_results = evaluate_model("XGBoost Baseline", xgb_baseline_pipeline, X_validation, y_validation)

print("✅ XGBoost baseline trained.")
for metric in ["Recall", "F1", "ROC_AUC", "Precision", "Accuracy"]:
    print(f"{metric}: {xgb_baseline_results[metric]:.4f}")

## 8. Lightweight XGBoost Optimization
This is intentionally small: 8 parameter combinations with 3-fold StratifiedKFold. It is fast, explainable, and realistic for the team.

In [ ]:
xgb_search_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("model", XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        subsample=0.9,
        colsample_bytree=0.9,
        random_state=RANDOM_STATE,
        n_jobs=1
    ))
])

param_grid = {
    "model__n_estimators": [100, 150],
    "model__max_depth": [3, 4],
    "model__learning_rate": [0.05, 0.10]
}

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

grid_search = GridSearchCV(
    estimator=xgb_search_pipeline,
    param_grid=param_grid,
    scoring={
        "Recall": "recall",
        "F1": "f1",
        "ROC_AUC": "roc_auc"
    },
    refit="Recall",
    cv=cv,
    n_jobs=1,
    verbose=1,
    return_train_score=False
)

grid_search.fit(X_train, y_train)

best_xgb_pipeline = grid_search.best_estimator_
xgb_optimized_results = evaluate_model("XGBoost Optimized", best_xgb_pipeline, X_validation, y_validation)

print("✅ Lightweight GridSearchCV complete.")
print(f"Best parameters: {grid_search.best_params_}")
print(f"Best CV recall: {grid_search.best_score_:.4f}")
for metric in ["Recall", "F1", "ROC_AUC", "Precision", "Accuracy"]:
    print(f"{metric}: {xgb_optimized_results[metric]:.4f}")

### ✅ GATE 3 — XGBoost Tuning Validation

In [ ]:
gate3_errors = []

total_grid_candidates = np.prod([len(v) for v in param_grid.values()])
if total_grid_candidates <= 8:
    print(f"✅ GATE 3 PASS: Grid is intentionally small ({total_grid_candidates} candidates).")
else:
    gate3_errors.append(f"❌ GATE 3 FAIL: Grid is too large ({total_grid_candidates} candidates).")

if isinstance(best_xgb_pipeline, Pipeline):
    print("✅ GATE 3 PASS: Best XGBoost model is an sklearn Pipeline.")
else:
    gate3_errors.append("❌ GATE 3 FAIL: Best XGBoost model is not a Pipeline.")

if hasattr(best_xgb_pipeline, "predict_proba"):
    print("✅ GATE 3 PASS: Best XGBoost Pipeline supports predict_proba.")
else:
    gate3_errors.append("❌ GATE 3 FAIL: Best XGBoost Pipeline cannot produce probabilities.")

if set(grid_search.scoring.keys()) == {"Recall", "F1", "ROC_AUC"} and grid_search.refit == "Recall":
    print("✅ GATE 3 PASS: GridSearchCV is recall-refit with F1 and ROC-AUC tracked.")
else:
    gate3_errors.append("❌ GATE 3 FAIL: GridSearchCV scoring/refit is incorrect.")

sample_proba = best_xgb_pipeline.predict_proba(X_validation.head(5))[:, 1]
if np.all((sample_proba >= 0) & (sample_proba <= 1)):
    print("✅ GATE 3 PASS: XGBoost probabilities are valid.")
else:
    gate3_errors.append("❌ GATE 3 FAIL: Invalid XGBoost probabilities.")

if gate3_errors:
    for err in gate3_errors:
        print(err)
    raise ValueError("❌ GATE 3 FAILED. Fix XGBoost setup before export.")

print("\n🟢 GATE 3 COMPLETE — XGBoost tuning is valid.")

## 9. Recall-First Model Comparison

In [12]:
comparison_records = []
for result in [lr_results, xgb_baseline_results, xgb_optimized_results]:
    comparison_records.append({
        "Model": result["Model"],
        "Recall": result["Recall"],
        "F1": result["F1"],
        "ROC_AUC": result["ROC_AUC"],
        "Precision": result["Precision"],
        "Accuracy": result["Accuracy"]
    })

comparison_df = (
    pd.DataFrame(comparison_records)
    .sort_values(["Recall", "F1", "ROC_AUC", "Precision", "Accuracy"], ascending=False)
    .reset_index(drop=True)
)

display(comparison_df.style.format({
    "Recall": "{:.4f}",
    "F1": "{:.4f}",
    "ROC_AUC": "{:.4f}",
    "Precision": "{:.4f}",
    "Accuracy": "{:.4f}"
}))

print("Evaluation priority: Recall → F1 → ROC-AUC → Precision → Accuracy")

,Model,Recall,F1,ROC_AUC,Precision,Accuracy
0,Logistic Regression,0.5722,0.6045,0.8483,0.6407,0.8013
1,XGBoost Optimized,0.5294,0.5815,0.8539,0.6450,0.7977
2,XGBoost Baseline,0.5267,0.5872,0.8559,0.6633,0.8034


Evaluation priority: Recall → F1 → ROC-AUC → Precision → Accuracy


## 10. Phase 4 Plots

In [ ]:
# Confusion matrix for optimized XGBoost
fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay.from_predictions(
    y_validation,
    xgb_optimized_results["y_pred"],
    display_labels=["Not Churned", "Churned"],
    cmap="Blues",
    values_format="d",
    ax=ax
)
ax.set_title("Confusion Matrix — Optimized XGBoost")
plt.tight_layout()
plt.savefig(CM_XGB_PATH, dpi=150, bbox_inches="tight")
plt.show()
print(f"📁 Saved: {CM_XGB_PATH}")

# ROC comparison: LR vs optimized XGBoost
plt.figure(figsize=(8, 6))
for label, result in [
    ("Logistic Regression", lr_results),
    ("XGBoost Optimized", xgb_optimized_results)
]:
    fpr, tpr, _ = roc_curve(y_validation, result["y_proba"])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, linewidth=2, label=f"{label} (AUC = {roc_auc:.3f})")

plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Random Guess")
plt.title("ROC Curve — Logistic Regression vs XGBoost")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend(loc="lower right")
plt.tight_layout()
plt.savefig(ROC_COMPARE_PATH, dpi=150, bbox_inches="tight")
plt.show()
print(f"📁 Saved: {ROC_COMPARE_PATH}")

# XGBoost feature importance
xgb_model = best_xgb_pipeline.named_steps["model"]
importance_df = (
    pd.DataFrame({
        "Feature": feature_schema,
        "Importance": xgb_model.feature_importances_
    })
    .sort_values("Importance", ascending=False)
    .reset_index(drop=True)
)

display(importance_df.head(15))

plot_df = importance_df.head(15).sort_values("Importance", ascending=True)
plt.figure(figsize=(10, 8))
plt.barh(plot_df["Feature"], plot_df["Importance"], color="#2a9d8f")
plt.title("Top XGBoost Feature Importances")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.tight_layout()
plt.savefig(XGB_IMPORTANCE_PATH, dpi=150, bbox_inches="tight")
plt.show()
print(f"📁 Saved: {XGB_IMPORTANCE_PATH}")

## 11. Select Final Model and Export Artifacts
The best overall model is selected using the project priority: Recall, then F1, then ROC-AUC.

In [ ]:
best_row = comparison_df.iloc[0]
best_model_name = best_row["Model"]

if best_model_name == "Logistic Regression":
    selected_model_template = lr_pipeline
elif best_model_name == "XGBoost Baseline":
    selected_model_template = xgb_baseline_pipeline
else:
    selected_model_template = best_xgb_pipeline

# Refit after validation-only selection. The untouched test is evaluated once below.
final_model_pipeline = clone(selected_model_template)
final_model_pipeline.fit(X_development, y_development)
xgb_export_pipeline = clone(best_xgb_pipeline)
xgb_export_pipeline.fit(X_development, y_development)
final_test_results = evaluate_model(f"{best_model_name} Final Holdout", final_model_pipeline, X_test, y_test)

backup_if_overwriting(XGB_MODEL_PATH, PROJECT_ROOT, "XGBoost model")
backup_if_overwriting(FINAL_MODEL_PATH, PROJECT_ROOT, "final selected model")
joblib.dump(xgb_export_pipeline, XGB_MODEL_PATH)
joblib.dump(final_model_pipeline, FINAL_MODEL_PATH)

print(f"Best validation model: {best_model_name}")
print("Untouched final holdout metrics:")
for metric in ["Recall", "F1", "ROC_AUC", "Precision", "Accuracy"]:
    print(f"{metric}: {final_test_results[metric]:.4f}")
print(f"📁 Saved XGBoost pipeline: {XGB_MODEL_PATH}")
print(f"📁 Saved final selected pipeline: {FINAL_MODEL_PATH}")
record_regeneration_step(
    PROJECT_ROOT,
    "04_xgboost_optimization",
    "notebooks/04_ali_xgboost_optimization.ipynb",
    [XGB_MODEL_PATH, FINAL_MODEL_PATH],
)


## 12. Export Phase 4 Comparison Report

In [ ]:
def markdown_table(df_to_render, float_cols=None):
    table = df_to_render.copy()
    if float_cols:
        for col in float_cols:
            if col in table.columns:
                table[col] = table[col].map(lambda x: f"{x:.4f}" if pd.notnull(x) else "")
    table = table.astype(str)
    headers = list(table.columns)
    lines = [
        "| " + " | ".join(headers) + " |",
        "| " + " | ".join(["---"] * len(headers)) + " |"
    ]
    for _, row in table.iterrows():
        lines.append("| " + " | ".join(str(row[col]).replace("|", "/") for col in table.columns) + " |")
    return "\n".join(lines)

metric_cols = ["Recall", "F1", "ROC_AUC", "Precision", "Accuracy"]

report_lines = [
    "# Phase 4 XGBoost Optimization Report",
    "",
    "## Source of Truth",
    "- Data: `data/cleaned/processed_telco.csv`",
    "- Feature schema: `models/feature_schema.pkl`",
    "- Phase 3 baseline: `models/logistic_regression_pipeline.pkl`",
    "- No raw data, preprocessing recreation, encoding recreation, or feature engineering recreation was used.",
    "",
    "## Optimization Strategy",
    "- Model: `XGBClassifier` inside an sklearn `Pipeline`",
    "- Search: very small `GridSearchCV`",
    "- CV: 3-fold stratified",
    "- Refit metric: Recall",
    "- Parameter grid size: 8 candidates",
    "- No SMOTE, resampling, threshold tuning, calibration, SHAP, Optuna, or ensembling.",
    "",
    "## Best XGBoost Parameters",
    "```text",
    str(grid_search.best_params_),
    "```",
    "",
    "## Validation Model Comparison",
    markdown_table(comparison_df, float_cols=metric_cols),
    "",
    "## Untouched Final Holdout",
    markdown_table(pd.DataFrame([{k: v for k, v in final_test_results.items() if k not in ["y_pred", "y_proba"]}]), float_cols=metric_cols),
    "",
    "## Final Model Decision",
    f"- Selected final model: `{best_model_name}`",
    "- Selection used validation metrics only. The untouched final holdout was evaluated once after refitting on train + validation.",
    "- Selection priority: Recall, then F1, then ROC-AUC.",
    "- If Logistic Regression remains best, that is an acceptable and honest modeling result.",
    "",
    "## Streamlit Handoff Notes",
    "- Use `models/final_model_pipeline.pkl` for the app unless the team explicitly chooses the XGBoost-only artifact.",
    "- Use `models/feature_schema.pkl` to create the exact 27 input columns in the exact order.",
    "- Do not manually scale inputs. Any needed transformations are inside saved pipelines.",
    "- Keep `processed_telco.csv` and `feature_schema.pkl` unchanged.",
    "",
    "## Exported Artifacts",
    "- `models/xgboost_pipeline.pkl`",
    "- `models/final_model_pipeline.pkl`",
    "- `assets/plots/models/confusion_matrix_xgb.png`",
    "- `assets/plots/models/roc_curve_xgb_vs_lr.png`",
    "- `assets/plots/models/feature_importance_xgb.png`",
    "",
    "## Final Verdict",
    "Phase 4 is complete and ready for the Streamlit application phase."
]

REPORT_PATH.write_text("\n".join(report_lines), encoding="utf-8")
print(f"📁 Saved: {REPORT_PATH}")

### ✅ GATE 4 — Export Validation

In [ ]:
gate4_errors = []

required_paths = [
    XGB_MODEL_PATH,
    FINAL_MODEL_PATH,
    REPORT_PATH,
    CM_XGB_PATH,
    ROC_COMPARE_PATH,
    XGB_IMPORTANCE_PATH
]

for path in required_paths:
    if path.exists() and path.stat().st_size > 0:
        print(f"✅ GATE 4 PASS: {path} exists and is non-empty.")
    else:
        gate4_errors.append(f"❌ GATE 4 FAIL: Missing or empty artifact: {path}")

loaded_xgb = joblib.load(XGB_MODEL_PATH)
loaded_final = joblib.load(FINAL_MODEL_PATH)

for name, model in [("XGBoost", loaded_xgb), ("Final Model", loaded_final)]:
    if isinstance(model, Pipeline) and hasattr(model, "predict_proba"):
        print(f"✅ GATE 4 PASS: Reloaded {name} is a probability-capable Pipeline.")
    else:
        gate4_errors.append(f"❌ GATE 4 FAIL: Reloaded {name} is not a valid Pipeline.")

    sample_pred = model.predict(X_validation.head(5))
    sample_proba = model.predict_proba(X_validation.head(5))[:, 1]
    if len(sample_pred) == 5 and np.all((sample_proba >= 0) & (sample_proba <= 1)):
        print(f"✅ GATE 4 PASS: Reloaded {name} predicts correctly on schema-aligned sample.")
    else:
        gate4_errors.append(f"❌ GATE 4 FAIL: Reloaded {name} prediction check failed.")

reloaded_schema = joblib.load(FEATURE_SCHEMA_PATH)
if reloaded_schema == feature_schema:
    print("✅ GATE 4 PASS: feature_schema.pkl remained unchanged.")
else:
    gate4_errors.append("❌ GATE 4 FAIL: feature_schema.pkl changed.")

if "Feature schema" in REPORT_PATH.read_text(encoding="utf-8"):
    print("✅ GATE 4 PASS: Report includes schema handoff notes.")
else:
    gate4_errors.append("❌ GATE 4 FAIL: Report missing schema handoff notes.")

if gate4_errors:
    for err in gate4_errors:
        print(err)
    raise ValueError("❌ GATE 4 FAILED. Fix exports before Phase 5 handoff.")

print("\n🟢 GATE 4 COMPLETE — Phase 4 exports validated.")
print("📬 HANDOFF READY → 05_ali_streamlit_app.ipynb")

## 13. Summary & Handoff

### Completed
- Loaded only `data/cleaned/processed_telco.csv`
- Enforced `models/feature_schema.pkl`
- Recreated the Phase 3 stratified split
- Evaluated the saved Logistic Regression baseline
- Trained an XGBoost baseline Pipeline
- Ran a very small recall-focused GridSearchCV
- Compared models using Recall, F1, ROC-AUC, Precision, and Accuracy
- Exported XGBoost and final selected model artifacts
- Exported confusion matrix, ROC comparison, feature importance, and report

### Required artifacts
- `models/xgboost_pipeline.pkl`
- `models/final_model_pipeline.pkl`
- `reports/xgboost_model_comparison.md`
- `assets/plots/models/confusion_matrix_xgb.png`
- `assets/plots/models/roc_curve_xgb_vs_lr.png`
- `assets/plots/models/feature_importance_xgb.png`

### Handoff to Phase 5
Phase 5 should use:
- `models/final_model_pipeline.pkl`
- `models/feature_schema.pkl`

Streamlit must create the same 27 feature columns in the exact schema order.
Do not manually scale inputs. Do not recreate training preprocessing.

> If XGBoost does not beat Logistic Regression, keep the honest result.
> A simpler baseline winning is acceptable in a clean ML project.